# Rebuild Derived Tables

Rebuilds gold/silver tables derived from upstream bronze/silver sources.  
Runs daily after the **Daily Data Refresh** job.

| Table | Source | Purpose |
| --- | --- | --- |
| `gold.macro_indicators_daily` | `bronze.fred_macro_indicators` | FRED macro data enriched with metadata |
| `gold.risk_factor_exposures` | `silver.stock_prices` | OLS market beta per stock |
| `silver.macro_indicators` | `bronze.macro_indicators_bronze` | Validated macro indicators |

In [0]:
# ── Import centralized config ────────────────────────────────────────
import sys, os
_nb  = dbutils.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_root = "/Workspace" + (_nb[:_nb.find("/notebooks/")] if "/notebooks/" in _nb else os.path.dirname(_nb))
sys.path.insert(0, _root)
from config import CATALOG, setup_logger, log_step

from pyspark.sql import functions as F
from pyspark.sql.window import Window

logger = setup_logger("riskbricks.rebuild_derived")

dbutils.widgets.text('catalog', CATALOG)
catalog = dbutils.widgets.get('catalog').strip()
spark.sql(f'USE CATALOG {catalog}')
print(f'Using catalog: {catalog}')


## 1. gold.macro_indicators_daily

In [0]:
print(f"Rebuilding gold.macro_indicators_daily from {catalog}.bronze.fred_macro_indicators...")

from config import FRED_INDICATOR_META
indicator_meta = FRED_INDICATOR_META

meta_rows = [(k, v[0], v[1], v[2], v[3]) for k, v in indicator_meta.items()]
meta_df = spark.createDataFrame(meta_rows, ["indicator", "series_title", "units", "frequency", "seasonal_adjustment"])

bronze = spark.table(f"{catalog}.bronze.fred_macro_indicators")

gold_macro = (
    bronze
    .join(meta_df, "indicator", "left")
    .withColumnRenamed("indicator", "indicator_name")
    .withColumn("realtime_start", F.col("date"))
    .withColumn("realtime_end", F.col("date"))
    .withColumn("notes", F.lit(None).cast("string"))
    .select(
        "date", "value", "indicator_name", "realtime_start", "realtime_end",
        "series_title", "units", "frequency", "seasonal_adjustment", "notes",
        "ingestion_timestamp",
    )
)

gold_macro.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.gold.macro_indicators_daily")

cnt = spark.table(f"{catalog}.gold.macro_indicators_daily").count()
latest = spark.sql(f"SELECT MAX(date) AS d FROM {catalog}.gold.macro_indicators_daily").first()["d"]
print(f"✅ gold.macro_indicators_daily: {cnt:,} rows, latest = {latest}")


## 2. gold.risk_factor_exposures

In [0]:
print(f"Rebuilding gold.risk_factor_exposures from {catalog}.silver.stock_prices (OLS beta)...")

prices = (
    spark.table(f"{catalog}.silver.stock_prices")
    .select("symbol", "date", "close")
    .filter(F.col("date") >= F.date_sub(F.current_date(), 365))
    .filter(F.col("close").isNotNull())
)

w = Window.partitionBy("symbol").orderBy("date")
returns = (
    prices
    .withColumn("prev_close", F.lag("close").over(w))
    .filter(F.col("prev_close").isNotNull())
    .withColumn("return", (F.col("close") - F.col("prev_close")) / F.col("prev_close"))
)

spy_returns = (
    returns.filter(F.col("symbol") == "SPY")
    .select(F.col("date"), F.col("return").alias("mkt_return"))
)

stock_vs_mkt = (
    returns.filter(F.col("symbol") != "SPY")
    .join(spy_returns, "date", "inner")
)

factor_stats = stock_vs_mkt.groupBy("symbol").agg(
    F.covar_samp("return", "mkt_return").alias("cov_im"),
    F.variance("mkt_return").alias("var_m"),
    F.variance("return").alias("var_i"),
    F.avg("return").alias("avg_return"),
    F.avg("mkt_return").alias("avg_mkt"),
    F.count("*").alias("n_obs"),
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date"),
)

factor_exposures = (
    factor_stats
    .filter(F.col("n_obs") >= 30)
    .withColumn("beta_mkt", F.col("cov_im") / F.col("var_m"))
    .withColumn("alpha", F.col("avg_return") - F.col("beta_mkt") * F.col("avg_mkt"))
    .withColumn("factor_var", F.col("beta_mkt") * F.col("beta_mkt") * F.col("var_m"))
    .withColumn("total_var", F.col("var_i"))
    .withColumn("idio_var", F.col("var_i") - F.col("factor_var"))
    .withColumn("total_vol", F.sqrt(F.col("var_i")))
    .withColumn("annualized_vol", F.sqrt(F.col("var_i")) * F.sqrt(F.lit(252)))
    .withColumn("model", F.lit("FF3-OLS"))
    .withColumn("beta_smb", F.lit(0.0))
    .withColumn("beta_hml", F.lit(0.0))
    .withColumn("computed_at", F.current_timestamp())
    .withColumn("start_date", F.col("start_date").cast("string"))
    .withColumn("end_date", F.col("end_date").cast("string"))
    .select(
        "symbol", "model", "alpha", "beta_mkt", "beta_smb", "beta_hml",
        "factor_var", "idio_var", "total_var", "total_vol", "annualized_vol",
        "start_date", "end_date", "computed_at",
    )
)

factor_exposures.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.gold.risk_factor_exposures")

cnt = spark.table(f"{catalog}.gold.risk_factor_exposures").count()
latest = spark.sql(f"SELECT MAX(end_date) AS d FROM {catalog}.gold.risk_factor_exposures").first()["d"]
print(f"✅ gold.risk_factor_exposures: {cnt:,} stocks, data through {latest}")


## 3. silver.macro_indicators

In [0]:
print(f"Rebuilding silver.macro_indicators from {catalog}.bronze.macro_indicators_bronze...")

bronze_macro = (
    spark.table(f"{catalog}.bronze.macro_indicators_bronze")
    .select("indicator_name", "date", "value")
    .filter(F.col("indicator_name").isNotNull())
    .filter(F.col("date").isNotNull())
    .filter(F.col("value").isNotNull())
    .withColumn("is_valid", F.lit(True))
    .withColumn("quality_score", F.lit(1.0))
    .withColumn("validated_at", F.current_timestamp())
)

w_dedup = Window.partitionBy("indicator_name", "date").orderBy(F.col("validated_at").desc())
validated = (
    bronze_macro
    .withColumn("rn", F.row_number().over(w_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

validated.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.macro_indicators")

cnt = spark.table(f"{catalog}.silver.macro_indicators").count()
latest = spark.sql(f"SELECT MAX(date) AS d FROM {catalog}.silver.macro_indicators").first()["d"]
print(f"✅ silver.macro_indicators: {cnt:,} rows, latest = {latest}")

## Summary

In [0]:
tables = [
    ("gold.macro_indicators_daily", "date"),
    ("gold.risk_factor_exposures", "end_date"),
    ("silver.macro_indicators", "date"),
]

print("=" * 60)
print("REBUILD DERIVED TABLES — COMPLETE")
print("=" * 60)

for tbl, date_col in tables:
    fqn = f"{catalog}.{tbl}"
    cnt = spark.table(fqn).count()
    latest = spark.sql(f"SELECT MAX({date_col}) AS d FROM {fqn}").first()["d"]
    print(f"  ✅ {tbl}: {cnt:,} rows, latest = {latest}")

print(f"\n  ⏱️  Completed at: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 60)